In [6]:
import json
path_json = "./all_entries_with_opcodes_df_en_cz_final_with_selected_8_docs.json"
# Load JSON data
with open(path_json, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

In [16]:
import pandas as pd
path_csv = "postedition_aligned_with_tokenized_offsets_df_en_cz_final_with_selected_8_docs.csv"
df_en_cz = pd.read_csv(path_csv)

In [21]:
import ast

df_en_cz['translation_tokenized'] = df_en_cz['translation_tokenized'].apply(ast.literal_eval)
df_en_cz['translation_word_offset'] = df_en_cz['translation_word_offset'].apply(ast.literal_eval)


### create the spans

In [29]:
## store the spans in each sentence
spans_ideal_translators = []

In [30]:
for index, row in df_en_cz.iterrows():
    spans_each_sentence_list = []
    
    spans = json_data[index]['difflib operations']
    if json_data[index]['relevant_words']:
        for span in spans:
            span_each_sentence = {}
            if span[0] == 'replace' or span[0] == 'delete':
                span_each_sentence['text'] = row["translation_tokenized"][span[1]: span[2]]
                span_each_sentence['start'] = row["translation_word_offset"][span[1]][0] 
                span_each_sentence['end'] = row["translation_word_offset"][span[2] -1][1]
           
            if span_each_sentence:
                spans_each_sentence_list.append(span_each_sentence)
    spans_ideal_translators.append(spans_each_sentence_list)

In [32]:
json_data[6]

{'index': 7,
 'source': "That system has already freed up more than 400 staff to go from manually checking transactions and records to client-facing roles where they can spend time helping customers, said Adrian Rigby, chief operating officer of HSBC's trade business.",
 'translation': 'Tento systém již uvolnil více než 400 zaměstnanců, kteří by mohli přecházet z ručních kontrol transakcí a záznamů do rolí zaměřených na klienta, kde mohou trávit čas pomáháním zákazníkům, uvedl Adrian Rigby, hlavní provozní ředitel obchodu HSBC.',
 'postedition': 'Tento systém umožnil již více než 400 zaměstnanců přechod od manuální kontroly transakcí a záznamů k funkcím zaměřeným na klienty, kde mohou více pomáhat zákazníkům, uvedl Adrian Rigby, hlavní výkonný obchodní ředitel HSBC.',
 'relevant_words': ['uvolnil',
  'zaměstnanců,',
  'kteří',
  'by',
  'mohli',
  'přecházet',
  'z',
  'ručních',
  'kontrol',
  'do',
  'rolí',
  'zaměřených',
  'klienta,',
  'trávit',
  'čas',
  'pomáháním',
  'provozn

In [33]:
import json

#spans_ideal_translators.json
with open("spans_ideal_df_en_cz_final_with_selected_8_docs.json", "w", encoding="utf-8") as f:
    json.dump(spans_ideal_translators, f, ensure_ascii=False, indent=2)


In [34]:
import copy

spans_ideal_translators_with_relevant_words = copy.deepcopy(spans_ideal_translators)


In [35]:
for index, row in df_en_cz.iterrows():
    relevant_words = json_data[index]['relevant_words']
    spans_ideal_translators_with_relevant_words[index].append(relevant_words)


In [36]:
import json
df_en_cz
#same as spans_ideal_translators_with_relevant_words.json
with open("spans_ideal_df_en_cz_final_with_selected_8_docs_with_relevant_words.json", "w", encoding="utf-8") as f:
    json.dump(spans_ideal_translators_with_relevant_words, f, ensure_ascii=False, indent=2)


### metadata for abstracts

In [37]:
import pandas as pd
df_abstracts = pd.read_csv("df_en_cz_final_with_selected_8_docs.csv")

In [41]:
df_abstracts.head(20)

,id_hal,Translation_id,line_id,source,translation,postedition
0,3,3,1,Using data and artificial intelligence to try ...,Využití dat a umělé inteligence k vyzkoušení a...,Využití dat a umělé inteligence ke zvýšení výn...
1,3,3,2,"""It's one of the first commercial uses of inve...",„Je to jedno z prvních komerčních investic do ...,„Je to jedna z prvních komerčních investic do ...
2,3,3,3,HSBC declined to comment on how much it expect...,"HSBC se odmítla vyjádřit k tomu, co očekává od...","HSBC se odmítla vyjádřit k tomu, co očekává od..."
3,3,3,4,The drive is an important part of the bank's e...,Pohon je důležitou součástí úsilí banky bránit...,Kampaň je důležitou součástí úsilí banky obháj...
4,3,3,5,HSBC was forced to invest hundreds of millions...,HSBC byla nucena investovat stovky milionů dol...,HSBC byla nucena investovat stovky milionů dol...
5,3,3,6,The system works by mapping individual custome...,"Systém funguje tak, že mapuje jednotlivé zákaz...","Systém funguje tak, že mapuje jednotlivé zákaz..."
6,3,3,7,That system has already freed up more than 400...,Tento systém již uvolnil více než 400 zaměstna...,Tento systém umožnil již více než 400 zaměstna...
7,3,3,8,HSBC's Nivison said the lightbulb moment was r...,"Společnost Nivison společnosti HSBC uvedla, že...","Pan Nivison z HSBC řekl, že si banka náhle uvě..."
8,5,5,1,Stem cells of 56 child cancer patients lost at...,Kmenové buňky 56 pacientů s rakovinou u dětí z...,Kmenové buňky 56 dětských pacientů s rakovinou...
9,5,5,2,A freezer malfunction at Children's Hospital L...,Porucha mrazáku v dětské nemocnici v Los Angel...,Porucha mrazáku v dětské nemocnici v Los Angel...


In [39]:
import pandas as pd

# First, convert numeric columns to appropriate types if needed
df_abstracts = df_abstracts.convert_dtypes()

# Create a groupby object for abstracts
abstract_groups = df_abstracts.groupby(['id_hal', 'Translation_id'])

# List to store abstract metadata
abstracts_metadata = []

# Process each abstract group
for (hal_id, trans_id), group in abstract_groups:
    # Calculate total characters in the abstract (sum of source sentence lengths)
    total_chars = group['translation'].str.len().sum()
    
    # Store metadata
    abstract_meta = {
        'hal_id': hal_id,
        'translation_id': trans_id,
        'sentence_indices': group.index.tolist(),
        'num_sentences': len(group),
        'total_characters': total_chars,
        'source': " ".join(group['source'].tolist())  # Optional: store actual text
    }
    abstracts_metadata.append(abstract_meta)

# Sort abstracts by total character count (descending order)
sorted_abstracts = sorted(abstracts_metadata, 
                         key=lambda x: x['total_characters'], 
                         reverse=True)

# Create sorted DataFrame by reordering based on sorted indices
sorted_indices = [idx for abstract in sorted_abstracts for idx in abstract['sentence_indices']]
sorted_df = df_abstracts.loc[sorted_indices].reset_index(drop=True)

# Optional: Convert metadata to DataFrame
metadata_df_original = pd.DataFrame(sorted_abstracts)

In [42]:
metadata_df_original.head(8)

,hal_id,translation_id,sentence_indices,num_sentences,total_characters,source
0,3,3,"[0, 1, 2, 3, 4, 5, 6, 7]",8,1785,Using data and artificial intelligence to try ...
1,7,7,"[24, 25, 26, 27, 28, 29, 30, 31]",8,1625,LNG investments hit record in 2019 & the bigge...
2,5,5,"[8, 9, 10, 11, 12, 13, 14, 15]",8,1575,Stem cells of 56 child cancer patients lost at...
3,13,13,"[32, 33, 34, 35, 36, 37, 38, 39]",8,1525,Jonathan Van Ness Just Hung Out With Nancy Pel...
4,18,18,"[48, 49, 50, 51, 52, 53, 54, 55]",8,1523,India's monsoon season has overrun by almost a...
5,17,17,"[40, 41, 42, 43, 44, 45, 46, 47]",8,1472,"""China has always been dedicated to resolving ..."
6,6,6,"[16, 17, 18, 19, 20, 21, 22, 23]",8,1367,Three Scottish students named among Europe's b...
7,19,19,"[56, 57, 58, 59, 60, 61, 62, 63]",8,1356,"US sends troops, air defense systems to Saudi ..."


In [43]:
# Initialize new columns in metadata_df
metadata_df_original['total_spans'] = 0
metadata_df_original['total_span_chars'] = 0

# Process each abstract group
for idx, row in metadata_df_original.iterrows():
    total_text_count = 0
    total_characters = 0
    
    # Iterate through each sentence index in the abstract
    for df_index in row['sentence_indices']:
        # Get corresponding JSON index (df index + 1)
        json_index = df_index
        
        # Get the JSON entry for this sentence
        sentence_entry = spans_ideal_translators[json_index]
        
        if sentence_entry:
            for span in sentence_entry:
                # due to /n/n after title, there is a mismatch of 1 position,
                # so >= is actually >=4
                if span['end'] - span['start'] >= 5:
                    # Count number of text elements
                    total_text_count += 1
                    
                    # Calculate span length (end - start)
                    total_characters += span['end'] - span['start']
    
    # Update metadata_df_original with calculated values
    metadata_df_original.at[idx, 'total_spans'] = total_text_count
    metadata_df_original.at[idx, 'total_span_chars'] = total_characters

In [45]:
metadata_df_original.head(8)

,hal_id,translation_id,sentence_indices,num_sentences,total_characters,source,total_spans,total_span_chars
0,3,3,"[0, 1, 2, 3, 4, 5, 6, 7]",8,1785,Using data and artificial intelligence to try ...,42,600
1,7,7,"[24, 25, 26, 27, 28, 29, 30, 31]",8,1625,LNG investments hit record in 2019 & the bigge...,29,401
2,5,5,"[8, 9, 10, 11, 12, 13, 14, 15]",8,1575,Stem cells of 56 child cancer patients lost at...,29,431
3,13,13,"[32, 33, 34, 35, 36, 37, 38, 39]",8,1525,Jonathan Van Ness Just Hung Out With Nancy Pel...,34,551
4,18,18,"[48, 49, 50, 51, 52, 53, 54, 55]",8,1523,India's monsoon season has overrun by almost a...,40,680
5,17,17,"[40, 41, 42, 43, 44, 45, 46, 47]",8,1472,"""China has always been dedicated to resolving ...",36,664
6,6,6,"[16, 17, 18, 19, 20, 21, 22, 23]",8,1367,Three Scottish students named among Europe's b...,30,574
7,19,19,"[56, 57, 58, 59, 60, 61, 62, 63]",8,1356,"US sends troops, air defense systems to Saudi ...",35,457


In [46]:
metadata_df_original.iloc[0]['sentence_indices']

[0, 1, 2, 3, 4, 5, 6, 7]

In [48]:
# Add ratio column using vectorized operations
metadata_df_original['span_char_ratio'] = metadata_df_original.apply(
    lambda row: (row['total_span_chars'] / row['total_characters']) 
                if row['total_characters'] > 0 
                else 0,
    axis=1
)

# Optional: Format as percentage with 2 decimal places
metadata_df_original['span_char_ratio_pct'] = (metadata_df_original['span_char_ratio'] * 100).round(2)

# Show results
print(metadata_df_original[['hal_id', 'total_characters', 'total_span_chars', 
                  'span_char_ratio', 'span_char_ratio_pct']])

   hal_id  total_characters  total_span_chars  span_char_ratio  \
0       3              1785               600         0.336134   
1       7              1625               401         0.246769   
2       5              1575               431         0.273651   
3      13              1525               551         0.361311   
4      18              1523               680         0.446487   
5      17              1472               664         0.451087   
6       6              1367               574         0.419898   
7      19              1356               457         0.337021   

   span_char_ratio_pct  
0                33.61  
1                24.68  
2                27.37  
3                36.13  
4                44.65  
5                45.11  
6                41.99  
7                33.70  


In [49]:
metadata_df_original = metadata_df_original.drop("span_char_ratio", axis=1)


In [52]:
metadata_df_original.head(8)

,hal_id,translation_id,sentence_indices,num_sentences,total_characters,source,total_spans,total_span_chars,span_char_ratio_pct
0,3,3,"[0, 1, 2, 3, 4, 5, 6, 7]",8,1785,Using data and artificial intelligence to try ...,42,600,33.61
1,7,7,"[24, 25, 26, 27, 28, 29, 30, 31]",8,1625,LNG investments hit record in 2019 & the bigge...,29,401,24.68
2,5,5,"[8, 9, 10, 11, 12, 13, 14, 15]",8,1575,Stem cells of 56 child cancer patients lost at...,29,431,27.37
3,13,13,"[32, 33, 34, 35, 36, 37, 38, 39]",8,1525,Jonathan Van Ness Just Hung Out With Nancy Pel...,34,551,36.13
4,18,18,"[48, 49, 50, 51, 52, 53, 54, 55]",8,1523,India's monsoon season has overrun by almost a...,40,680,44.65
5,17,17,"[40, 41, 42, 43, 44, 45, 46, 47]",8,1472,"""China has always been dedicated to resolving ...",36,664,45.11
6,6,6,"[16, 17, 18, 19, 20, 21, 22, 23]",8,1367,Three Scottish students named among Europe's b...,30,574,41.99
7,19,19,"[56, 57, 58, 59, 60, 61, 62, 63]",8,1356,"US sends troops, air defense systems to Saudi ...",35,457,33.70


In [51]:
#same as - abstract_metadata_with_ideal_spans_correct_no_spans_with_min4chars.tsv
metadata_df_original.to_csv('abstract_metadata_with_ideal_spans_correct_no_spans_with_min4chars_df_en_cz_final_with_selected_8_docs.tsv', sep='\t', index=False)

**take 2 sets of 4 abstracts so that we get mean 1500 characters**

In [54]:
from itertools import combinations

ids = metadata_df_original["translation_id"].tolist()
chars = dict(zip(metadata_df_original["translation_id"], metadata_df_original["total_characters"]))

target = 1500 * 4  # sum needed for 4 items to average 1500

best_split = None
best_diff = float("inf")

for combo in combinations(ids, 4):
    group1 = combo
    group2 = [i for i in ids if i not in combo]
    diff = abs(sum(chars[i] for i in group1) - target) + abs(sum(chars[i] for i in group2) - target)
    if diff < best_diff:
        best_diff = diff
        best_split = (group1, group2)

print("Group 1:", best_split[0], "mean:", sum(chars[i] for i in best_split[0]) / 4)
print("Group 2:", best_split[1], "mean:", sum(chars[i] for i in best_split[1]) / 4)

Group 1: (3, 7, 6, 19) mean: 1533.25
Group 2: [5, 13, 18, 17] mean: 1523.75


In [55]:
spans = dict(zip(metadata_df_original["translation_id"], metadata_df_original["total_spans"]))

group1 = (3, 7, 6, 19)
group2 = (5, 13, 18, 17)

print("Group 1 avg spans:", sum(spans[i] for i in group1) / len(group1))
print("Group 2 avg spans:", sum(spans[i] for i in group2) / len(group2))

Group 1 avg spans: 34.0
Group 2 avg spans: 34.75
